### Nota: Esta notebook fue generado en su mayoría con IA, para traducir lógica de SQL a python lo más rápdio posible. 

### Bajar al final de la query para features transformadas para el modelo fuera de la query

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

orders = pd.read_csv("../data/orders.csv", parse_dates=[
    "checkout_timestamp", "merchant_notification_time",
    "rider_notification_time", "rider_arrival_time",
    "out_for_delivery_time", "delivery_timestamp"
])
print(f"Órdenes cargadas: {len(orders)}")

Órdenes cargadas: 10000


In [16]:
# Equivalente al CTE order_segments del SQL.

orders["total_delivery_min"] = (
    orders.delivery_timestamp - orders.checkout_timestamp
).dt.total_seconds() / 60

orders["rider_wait_min"] = (
    orders.out_for_delivery_time - orders.rider_arrival_time
).dt.total_seconds() / 60

orders["obs_prep_min"] = (
    orders.out_for_delivery_time - orders.merchant_notification_time
).dt.total_seconds() / 60

orders["rider_dispatch_min"] = (
    orders.rider_arrival_time - orders.rider_notification_time
).dt.total_seconds() / 60

orders["delivery_leg_min"] = (
    orders.delivery_timestamp - orders.out_for_delivery_time
).dt.total_seconds() / 60

orders["checkout_hour"] = orders.checkout_timestamp.dt.hour
orders["day_of_week"]   = orders.checkout_timestamp.dt.dayofweek  # 0=lunes, 6=domingo

# Filtro equivalente al WHERE del SQL
orders = orders[orders.total_delivery_min.between(5, 180)].copy()
orders = orders.sort_values("checkout_timestamp").reset_index(drop=True)

print(f"Órdenes después del filtro: {len(orders)}")

Órdenes después del filtro: 10000


In [17]:
# Equivalente al CTE merchant_prep_time_stats.

WINDOW_DAYS       = 30
RIDER_WAIT_THRESH = 2

merchant_stats_rows = []

for merchant_id, group in orders.groupby("merchant_id"):
    group = group.sort_values("checkout_timestamp").reset_index(drop=True)

    for i, row in group.iterrows():
        cutoff = row.checkout_timestamp - pd.Timedelta(days=WINDOW_DAYS)

        hist = group[
            (group.checkout_timestamp < row.checkout_timestamp) &
            (group.checkout_timestamp >= cutoff)
        ]
        reliable = hist[hist.rider_wait_min > RIDER_WAIT_THRESH]

        merchant_stats_rows.append({
            "order_id":                  row.order_id,
            "merchant_avg_prep_min":     reliable.obs_prep_min.mean() if len(reliable) > 0 else np.nan,
            "merchant_std_prep_min":     reliable.obs_prep_min.std()  if len(reliable) > 1 else np.nan,
            "merchant_pct_rider_waits":  (hist.rider_wait_min > RIDER_WAIT_THRESH).mean()
                                         if len(hist) > 0 else np.nan,
        })

merchant_stats = pd.DataFrame(merchant_stats_rows)
print(f"merchant_stats generado: {len(merchant_stats)} filas")
print(f"NaNs en merchant_avg_prep_min: {merchant_stats.merchant_avg_prep_min.isna().sum()}")


merchant_stats generado: 10000 filas
NaNs en merchant_avg_prep_min: 409


In [18]:
# Equivalente al CTE zone_stats.

zone_stats_rows = []

for (zone_id, hour), group in orders.groupby(["zone_id", "checkout_hour"]):
    group = group.sort_values("checkout_timestamp").reset_index(drop=True)

    for i, row in group.iterrows():
        cutoff = row.checkout_timestamp - pd.Timedelta(days=WINDOW_DAYS)

        hist = group[
            (group.checkout_timestamp < row.checkout_timestamp) &
            (group.checkout_timestamp >= cutoff)
        ]

        zone_stats_rows.append({
            "order_id":                    row.order_id,
            "zone_avg_delivery_min":       hist.delivery_leg_min.mean()   if len(hist) > 0 else np.nan,
            "zone_avg_rider_dispatch_min": hist.rider_dispatch_min.mean() if len(hist) > 0 else np.nan,
        })

zone_stats = pd.DataFrame(zone_stats_rows)
print(f"zone_stats generado: {len(zone_stats)} filas")
print(f"NaNs en zone_avg_delivery_min: {zone_stats.zone_avg_delivery_min.isna().sum()}")

zone_stats generado: 10000 filas
NaNs en zone_avg_delivery_min: 113


In [19]:
# Equivalente al CTE demand_stats.

demand_rows = []

for merchant_id, group in orders.groupby("merchant_id"):
    for i, row in group.iterrows():
        ongoing = group[
            (group.checkout_timestamp <= row.checkout_timestamp) &
            (group.out_for_delivery_time > row.checkout_timestamp) &
            (group.order_id != row.order_id)
        ]
        demand_rows.append({
            "order_id":               row.order_id,
            "merchant_orders_ongoing": len(ongoing),
        })

demand_stats = pd.DataFrame(demand_rows)
print(f"demand_stats generado: {len(demand_stats)} filas")
print(demand_stats.merchant_orders_ongoing.describe().round(1))

demand_stats generado: 10000 filas
count    10000.0
mean         0.1
std          0.2
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          3.0
Name: merchant_orders_ongoing, dtype: float64


In [20]:
# Equivalente al CTE category_defaults.

category_defaults = (
    orders[orders.rider_wait_min > RIDER_WAIT_THRESH]
    .groupby("order_category")
    .agg(
        avg_prep_min         =("obs_prep_min",      "mean"),
        std_prep_min         =("obs_prep_min",      "std"),
        avg_pct_rider_waits  =("rider_wait_min",    lambda x: (x > RIDER_WAIT_THRESH).mean()),
        avg_delivery_min     =("delivery_leg_min",  "mean"),
        avg_dispatch_min     =("rider_dispatch_min","mean"),
    )
    .reset_index()
)
print(category_defaults)

  order_category  avg_prep_min  std_prep_min  avg_pct_rider_waits  \
0           food     35.025293     16.283040                  1.0   
1       pharmacy     19.521592      6.430536                  1.0   
2    supermarket     24.554040      9.437415                  1.0   

   avg_delivery_min  avg_dispatch_min  
0         18.836735         14.359170  
1         18.624476         11.437000  
2         18.820216         12.847635  


In [21]:
# Equivalente al SELECT final con JOINs y COALESCEs.

dataset = (
    orders
    .merge(merchant_stats, on="order_id", how="left")
    .merge(zone_stats,     on="order_id", how="left")
    .merge(demand_stats,   on="order_id", how="left")
    .merge(category_defaults, on="order_category", how="left")
)

# Aplicar fallbacks (equivalente a COALESCE)
dataset["merchant_avg_prep_min"]     = dataset["merchant_avg_prep_min"].fillna(dataset["avg_prep_min"])
dataset["merchant_std_prep_min"]     = dataset["merchant_std_prep_min"].fillna(dataset["std_prep_min"])
dataset["merchant_pct_rider_waits"]  = dataset["merchant_pct_rider_waits"].fillna(dataset["avg_pct_rider_waits"])
dataset["zone_avg_delivery_min"]     = dataset["zone_avg_delivery_min"].fillna(dataset["avg_delivery_min"])
dataset["zone_avg_rider_dispatch_min"] = dataset["zone_avg_rider_dispatch_min"].fillna(dataset["avg_dispatch_min"])
dataset["merchant_orders_ongoing"]   = dataset["merchant_orders_ongoing"].fillna(0)

### Features transformadas para el modelo

In [22]:
# Cyclical encoding para la hora
dataset["sin_hour"] = np.sin(2 * np.pi * dataset["checkout_hour"] / 24)
dataset["cos_hour"] = np.cos(2 * np.pi * dataset["checkout_hour"] / 24)

# OHE para categoría y día de la semana
dataset = pd.get_dummies(dataset, columns=["order_category", "day_of_week"], drop_first=True)

FEATURE_COLS = [
    "sin_hour", "cos_hour",
    "distance_km",
    "merchant_avg_prep_min", "merchant_std_prep_min", "merchant_pct_rider_waits",
    "zone_avg_delivery_min", "zone_avg_rider_dispatch_min",
    "merchant_orders_ongoing",
] + [c for c in dataset.columns if c.startswith("order_category_") or c.startswith("day_of_week_")]

TARGET_COL = "total_delivery_min"

print(f"Features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Features: 17
['sin_hour', 'cos_hour', 'distance_km', 'merchant_avg_prep_min', 'merchant_std_prep_min', 'merchant_pct_rider_waits', 'zone_avg_delivery_min', 'zone_avg_rider_dispatch_min', 'merchant_orders_ongoing', 'order_category_pharmacy', 'order_category_supermarket', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6']


In [27]:
# Usamos split temporal, NO aleatorio.
# Entrenar en datos futuros y testear en pasados sería data leakage.

cutoff_train = pd.Timestamp("2026-02-28")
cutoff_val   = pd.Timestamp("2026-03-16")

train = dataset[dataset.checkout_timestamp <  cutoff_train]
val   = dataset[(dataset.checkout_timestamp >= cutoff_train) & (dataset.checkout_timestamp < cutoff_val)]
test  = dataset[dataset.checkout_timestamp >= cutoff_val]

print(f"Train: {len(train)} ({len(train)/len(dataset)*100:.0f}%)")
print(f"Val:   {len(val)}   ({len(val)/len(dataset)*100:.0f}%)")
print(f"Test:  {len(test)}  ({len(test)/len(dataset)*100:.0f}%)")

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

Train: 6517 (65%)
Val:   1769   (18%)
Test:  1714  (17%)


In [28]:
import joblib

train[FEATURE_COLS + [TARGET_COL]].to_csv("../data/train.csv", index=False)
val[FEATURE_COLS   + [TARGET_COL]].to_csv("../data/val.csv",   index=False)
test[FEATURE_COLS  + [TARGET_COL]].to_csv("../data/test.csv",  index=False)

joblib.dump(FEATURE_COLS, "../model/feature_cols.pkl")

print("Datasets guardados en data/")

Datasets guardados en data/
